# Phase 4 - Feature engineering

Thin notebook that orchestrates `seercast.training.build_features` to:

1. Read `data/interim/m5_base_ca1.parquet`.
2. Validate the base table (strict).
3. Build the supervised table for the direct horizons `[1, 7, 14, 28]` and persist to `data/processed/train_features_ca1.parquet`.
4. Build the full-horizon table (1..28) for Phase 7 scenario simulation and persist to `data/processed/train_features_ca1_full_horizon.parquet`.
5. Print the validation report and a missing-feature-rate summary.

All real logic is in `seercast.features.{calendar_features, demand_features, price_features, supervised}`. Leakage rules are enforced by construction (per-id `shift(1)` before any rolling stat) and verified by the validator.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.training.build_features import run as build_features
from seercast.features import SUPERVISED_COLUMNS, validate_supervised_table

REPO_ROOT

## 1. Build both supervised tables

In [ ]:
tables = build_features()
direct = tables['direct']
full = tables['full']
print('direct:', direct.shape)
print('full  :', full.shape)
direct.head()

## 2. Schema sanity check

All canonical supervised columns should be present, in canonical order.

In [ ]:
missing = [c for c in SUPERVISED_COLUMNS if c not in direct.columns]
extra = [c for c in direct.columns if c not in SUPERVISED_COLUMNS]
print('schema missing:', missing or 'none')
print('schema extras :', extra or 'none')

## 3. Missing-feature-rate breakdown

Some missingness is expected: `sales_lag_56` is NaN for the first 56 days per id, `price_lag_7` is NaN if the item wasn't priced 7 days earlier, etc. Anything above its tolerance shows up as a warning in the validator above.

In [ ]:
report = validate_supervised_table(direct)
miss = pd.Series(report.missing_feature_rates).sort_values(ascending=False)
miss = miss[miss > 0]
miss

## 4. Eyeball one (id, origin_date) row

All 4 horizons should appear, target_date should equal origin_date + horizon days, and the same origin features should repeat across horizons (those are 'as-of origin').

In [ ]:
sample_id = direct['id'].iloc[0]
sample_origin = direct['origin_date'].iloc[0]
sample = direct[(direct['id'] == sample_id) & (direct['origin_date'] == sample_origin)]
sample[['id', 'origin_date', 'horizon', 'target_date', 'target_sales',
        'sales_lag_1', 'sales_lag_7', 'rolling_mean_28',
        'sell_price', 'target_sell_price',
        'target_dayofweek', 'target_is_weekend', 'target_has_event', 'target_is_snap_day']]

**Next:** Phase 5 - LightGBM point model with the same rolling-origin backtester from Phase 3.